# Dataset Analysis

This notebook performs exploratory analysis on the fruit image dataset.
The goal is to understand dataset structure, class distribution, and potential labeling issues,
in preparation for model training experiments.


Imports & Global Settings

In [ ]:
import os
import random
from collections import Counter
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import drive

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["axes.grid"] = False


Dataset Paths & Basic Structure

In [ ]:
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/CA Notebook/image"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")
print("Train samples:", len(os.listdir(TRAIN_DIR)))
print("Test samples :", len(os.listdir(TEST_DIR)))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train samples: 240
Test samples : 60


## Dataset Structure

The dataset is organized into train and test folders.
Class labels are inferred from image file names using the format: className_index.


Parse Labels from Filenames

In [ ]:
def get_label_from_filename(filename):
    return filename.split("_")[0].lower()


Class Distribution Analysis

Count samples per class

In [ ]:
train_files = os.listdir(TRAIN_DIR)
train_labels = [get_label_from_filename(f) for f in train_files]

class_counts = Counter(train_labels)
class_counts


Counter({'apple': 75, 'banana': 73, 'orange': 72, 'mixed': 20})

Plot Class Distribution

In [ ]:
classes = list(class_counts.keys())
counts = list(class_counts.values())

plt.figure(figsize=(6, 5))
bars=plt.bar(classes, counts,width=0.5)
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.title("Class Distribution in Training Set")
plt.bar_label(bars, padding=2)
plt.tight_layout()

plt.savefig("class_distribution_train.png", dpi=300)
plt.show()


Sample Visualization

In [ ]:
import matplotlib.pyplot as plt
import random
import os
from PIL import Image

def show_all_samples_aligned(class_counts, n=3, figsize_width=10):
    classes = sorted(class_counts.keys())
    num_classes = len(classes)

    # 1. Create a unified canvas
    # The first parameter of figsize controls the total width (decrease it to narrow the plot), the second parameter adjusts the height automatically based on the number of classes
    fig, axes = plt.subplots(num_classes, n, figsize=(figsize_width, num_classes * 4))

    # Compatibility for single class case
    if num_classes == 1: axes = [axes]

    for row_idx, cls in enumerate(classes):
        # 2. Get random samples for the current class
        files = [f for f in train_files if get_label_from_filename(f) == cls]
        samples = random.sample(files, min(n, len(files)))

        for col_idx in range(n):
            ax = axes[row_idx][col_idx]

            if col_idx < len(samples):
                fname = samples[col_idx]
                img = Image.open(os.path.join(TRAIN_DIR, fname))
                ax.imshow(img)

                # Add filename below the image
                ax.set_xlabel(fname, fontsize=9)

            # Hide ticks while keeping labels visible
            ax.set_xticks([])
            ax.set_yticks([])
            # Remove black borders
            for spine in ax.spines.values():
                spine.set_visible(False)

            # 3. Display class title above the middle image of each row
            if col_idx == n // 2:
                ax.set_title(f"Samples from class: {cls}",
                             fontsize=14, pad=25, fontweight='bold', color='darkblue')

        # 4. Draw dividing lines between rows (excluding the last row)
        if row_idx < num_classes - 1:
            # Calculate y-coordinate of the dividing line (in figure coordinate system, 0 to 1 from bottom to top)
            # Precisely position by adjusting the spacing after subplots_adjust
            line_y = 1 - ((row_idx + 1) / num_classes) + 0.02
            line = plt.Line2D([0.05, 0.95], [line_y, line_y],
                              transform=fig.transFigure, color="lightgray",
                              linestyle="--", linewidth=1.5)
            fig.add_artist(line)

    # Adjust layout: increase row spacing with hspace to prevent title overlapping with the upper row
    plt.tight_layout()
    plt.subplots_adjust(hspace=0.6)
    plt.show()

# Calling method
show_all_samples_aligned(class_counts, n=3, figsize_width=8)

Mislabel Inspection

In [ ]:
suspected_mislabels = [
    "banana_35.jpg",
    "banana_61.jpg",
]

plt.figure(figsize=(6, 3))
for i, fname in enumerate(suspected_mislabels):
    img = Image.open(os.path.join(TRAIN_DIR, fname))
    plt.subplot(1, len(suspected_mislabels), i + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(fname, fontsize=8)

plt.suptitle("Examples of Mislabelled Images")
plt.show()


In [ ]:
train_files = os.listdir(TRAIN_DIR)
train_labels = [get_label_from_filename(f) for f in train_files]

class_counts = Counter(train_labels)
print(class_counts)

Counter({'apple': 75, 'orange': 72, 'banana': 71, 'mixed': 22})


In [ ]:
classes = list(class_counts.keys())
counts = list(class_counts.values())

plt.figure(figsize=(6, 5))
bars=plt.bar(classes, counts,width=0.5)
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.title("Class Distribution in Training Set(After modifying wrong labels)")
plt.bar_label(bars, padding=2)
plt.tight_layout()

plt.savefig("class_distribution_train.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Prepare data
data_before = {'apple': 75, 'banana': 73, 'orange': 72, 'mixed': 20}
data_after = {'apple': 75, 'orange': 72, 'banana': 71, 'mixed': 22}

# Unify the order of categories to ensure accurate comparison
labels = ['apple', 'banana', 'orange', 'mixed']
counts_before = [data_before[label] for label in labels]
counts_after = [data_after[label] for label in labels]

x = np.arange(len(labels))  # Positions for category labels
width = 0.35  # Width of each bar

# 2. Create the plot
fig, ax = plt.subplots(figsize=(8, 5))

# Plot bars for before and after correction
# Use soft Morandi/pastel color palette
rects1 = ax.bar(x - width/2, counts_before, width, label='Original Labels', color='#A2CFFE') # Pastel blue
rects2 = ax.bar(x + width/2, counts_after, width, label='Corrected Labels', color='#FFB3BA')  # Coral pink

# 3. Add text annotations
ax.set_ylabel('Number of Images', fontsize=11)
ax.set_title('Label Distribution Comparison (Before vs After Correction)', pad=20, fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

# 4. Label values on top of the bars
ax.bar_label(rects1, padding=3)
ax.bar_label(rects2, padding=3)

# 5. Beautify: Remove top and right spines for a cleaner look
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.tight_layout()
plt.show()

Image size analysis

In [ ]:
import numpy as np
from tqdm import tqdm

def analyze_image_resolutions(image_dir, image_files):
    widths = []
    heights = []

    print("Image size is being measured....")
    for fname in tqdm(image_files):
        img_path = os.path.join(image_dir, fname)
        try:
            # Using Image.open().size only reads metadata and does not load image pixels, making it extremely fast.
            with Image.open(img_path) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)
        except Exception as e:
            print(f"Cannot read file  {fname}: {e}")

    # 1. Calculate average
    avg_width = np.mean(widths)
    avg_height = np.mean(heights)

    print(f"\nResult:")
    print(f"Average Width: {avg_width:.2f} px")
    print(f"Average Height: {avg_height:.2f} px")
    print(f"Size range: ({min(widths)}x{min(heights)}) to ({max(widths)}x{max(heights)})")

    # 2. Draw a two-dimensional distribution point map
    plt.figure(figsize=(8, 6))

    # Use alpha to control transparency; overlapping points will appear darker, reflecting distribution density.
    plt.scatter(widths, heights, alpha=0.5, c='royalblue', edgecolors='none', s=30)

    # Plot the average value (red crosshairs).
    plt.axvline(avg_width, color='red', linestyle='--', alpha=0.6, label=f'Avg Width: {avg_width:.1f}')
    plt.axhline(avg_height, color='green', linestyle='--', alpha=0.6, label=f'Avg Height: {avg_height:.1f}')

    plt.xlabel('Width (pixels)')
    plt.ylabel('Height (pixels)')
    plt.title('Image Resolution Distribution')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend()

    plt.tight_layout()
    plt.show()

analyze_image_resolutions(TRAIN_DIR, train_files)